# LifeMate Vast — đánh giá checkpoint 1524 và thử warm-start có kiểm soát

Chỉ cần upload **notebook này** và file `lifemate-resume-latest.tar.gz` (checkpoint 1524) lên Vast, chọn kernel Python 3.12+ rồi Run All từ trên xuống.

**Notebook làm gì**
1. Tải bundle code + dữ liệu cũ, nhập checkpoint 1524, **sao lưu 1524 sang thư mục riêng** trước mọi thứ khác.
2. Áp bản code đã sửa (có SHA-256 kiểm tra, giữ bản gốc `.orig`).
3. Đo dev baseline 1524 và lưu báo cáo từng mẫu **trước khi train**.
4. Chạy theo `MODE` ở cell cấu hình:
   - `warmstart` *(mặc định)*: run **mới và ngắn** từ trọng số adapter 1524, optimizer/scheduler mới, LR thấp giảm dần về 0. Ghi ra thư mục riêng, không đụng `full/`.
   - `continue`: tiếp tục đúng run cũ từ 1524 (giống notebook trước). Run này sẽ tự dừng sau ~1–2 lần eval vì early stopping vẫn so với loss tốt nhất 0,03678.
   - `eval_only`: chỉ đo dev; mặc định chọn baseline.
5. Đo dev rollout cho adapter ứng viên; chọn `PUBLIC_CHOICE` tường minh. Nếu ứng viên kém baseline state-match quá ngưỡng, public inference bị chặn.
6. Đóng gói kết quả để tải về.

**Lưu ý trung thực:** `warmstart` với LR 2e-5 là *giả thuyết cần đo* (lịch LR cũ gần như không giảm), không phải cấu hình đã tối ưu. Dev chỉ có 154 mẫu; sai số xấp xỉ ±4 điểm phần trăm. Call-F1 là chỉ số diagnostic tự tính, chưa chứng minh trùng công thức BTC và không dùng để kết luận kỹ thuật nào làm tăng điểm.

**Thời gian (ước tính, chưa đo trên GPU của bạn):** các checkpoint đầu cho thấy khoảng 30 giây/bước tối ưu hoá, tức ~4–5 giờ cho 508 bước. Đo dev 154 mẫu mỗi adapter thêm một khoảng nữa; cache ASR giúp lần đo thứ hai và public nhanh hơn.


In [ ]:
from pathlib import Path
import os

# --- Bundle code + dữ liệu cơ sở (đã dùng ở các notebook trước) ---
RELEASE_REPO = "Vincent-1512/ZALO_AI_CHALLENGE_2026"
RELEASE_TAG = "lifemate-resume-508-20261107"
BUNDLE_NAME = "lifemate-vast-resume-clean-508-20261107.tar.gz"
BUNDLE_SHA256 = "b7413e49c002a7afb6fea52889da370b7961a3e9c847201dac0ff0202f764bbf"
WORKSPACE_DIR = Path("/workspace")  # nếu đã có thư mục lifemate/ khác, chọn thư mục mới

# --- Gói checkpoint-1524 ---
# Notebook tự tải file này từ GitHub Release bên dưới (kiểm SHA-256). Nếu Release chưa có hoặc tải lỗi,
# upload tay file này lên /workspace rồi chạy lại.
HANDOFF_ARCHIVE = Path("/workspace/lifemate-resume-latest.tar.gz")
HANDOFF_NAME = "lifemate-resume-latest.tar.gz"
HANDOFF_SHA256 = "8849e69703fc2a62dd084b15b8ea2f1803b6fd4b8b193d2cf224518f27048b4c"
HANDOFF_RELEASE_TAG = "lifemate-resume-1524-20261109"  # để "" nếu chỉ muốn upload tay

# --- Chế độ chạy ---
MODE = "warmstart"      # "warmstart" | "continue" | "eval_only"
WARM_LR = 2e-5          # giả thuyết cần đo; đổi MỘT thông số mỗi lần nếu thử lại
WARM_MAX_STEPS = 508    # 1 epoch dữ liệu augmentation; cosine giảm về ~0 trong đúng số bước này
WARM_SEED = 2027        # khác 2026 để không lặp lại đúng thứ tự dữ liệu epoch đầu của run cũ
SMOKE_TEST = True       # smoke 2 bước warm-start trước run ngắn

EVAL_BASELINE = True    # đo dev cho 1524 (cần để so sánh; dùng cache ASR nên lần sau nhanh hơn)
RUN_PUBLIC = True       # tạo public submission cho adapter được chọn
PUBLIC_CHOICE = "baseline" if MODE == "eval_only" else "candidate"  # chọn tường minh; không tự xếp hạng adapter
SKIP_PUBLIC_IF_WORSE_BY = 0.03  # chặn nếu candidate state-match kém baseline >3 điểm %; None để tắt

# --- Thông số run gốc (không đổi: dùng để tái tạo đúng dữ liệu và kiểm hash) ---
RUN_ID = "vast-20261106-124816"
SEED = 2026
LR_ORIG = 5e-5
MAX_EPOCHS = 30
MAX_SEQUENCE_LENGTH = 8192
GRAD_ACCUMULATION = 16
LORA_RANK, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
EVALS_PER_EPOCH = 2
EARLY_STOPPING_PATIENCE = 4
EARLY_STOPPING_THRESHOLD = 0.001
SCHEMA_PERMUTATIONS = 1

assert MODE in {"warmstart", "continue", "eval_only"}
assert PUBLIC_CHOICE in {"baseline", "candidate"}
assert SKIP_PUBLIC_IF_WORSE_BY is None or SKIP_PUBLIC_IF_WORSE_BY >= 0
print("MODE =", MODE)
print("PUBLIC_CHOICE =", PUBLIC_CHOICE)


## 1. Tải code + dữ liệu cơ sở và nhập checkpoint 1524

In [ ]:
"""Download a pinned GitHub release bundle for the Vast notebook."""
from __future__ import annotations

import hashlib
import json
import os
import shutil
import tarfile
import tempfile
import time
import urllib.error
import urllib.request
from pathlib import Path


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def api_request(url: str, token: str, binary: bool = False, offset: int = 0):
    headers = {
        "Accept": "application/octet-stream" if binary else "application/vnd.github+json",
        "User-Agent": "lifemate-vast-bootstrap",
        "X-GitHub-Api-Version": "2022-11-28",
    }
    if offset:
        headers["Range"] = f"bytes={offset}-"
    request = urllib.request.Request(url, headers=headers)
    # urllib does not forward unredirected headers to the asset CDN.
    if token:
        request.add_unredirected_header("Authorization", f"Bearer {token}")
    return urllib.request.urlopen(request, timeout=90)


def get_release_asset(repo: str, tag: str, filename: str, token: str) -> dict:
    base = f"https://api.github.com/repos/{repo}"
    with api_request(f"{base}/releases/tags/{tag}", token) as response:
        release = json.load(response)
    assets = [asset for asset in release.get("assets", []) if asset["name"] == filename]
    if len(assets) != 1 or assets[0].get("state") != "uploaded":
        raise FileNotFoundError(f"Release {tag} chưa có asset hoàn chỉnh: {filename}")
    return assets[0]


def download_once(repo: str, asset: dict, partial: Path, token: str) -> None:
    offset = partial.stat().st_size if partial.exists() else 0
    url = f"https://api.github.com/repos/{repo}/releases/assets/{int(asset['id'])}"
    with api_request(url, token, binary=True, offset=offset) as response:
        append = response.status == 206 and offset > 0
        if append and not response.headers.get("Content-Range", "").startswith(f"bytes {offset}-"):
            raise RuntimeError("Server trả sai vị trí download tiếp tục.")
        downloaded = offset if append else 0
        last_print = downloaded
        with partial.open("ab" if append else "wb") as stream:
            while chunk := response.read(1024 * 1024):
                stream.write(chunk)
                downloaded += len(chunk)
                if downloaded - last_print >= 64 * 1024**2:
                    print(f"Downloaded {downloaded / 1024**2:.0f}/{asset['size'] / 1024**2:.0f} MiB", flush=True)
                    last_print = downloaded


def download_asset(repo: str, asset: dict, target: Path, digest: str, token: str) -> None:
    if target.is_file() and file_sha256(target) == digest:
        return
    partial = target.with_suffix(target.suffix + ".part")
    for attempt in range(3):
        try:
            if not partial.exists() or partial.stat().st_size != asset["size"]:
                download_once(repo, asset, partial, token)
            if partial.stat().st_size != asset["size"] or file_sha256(partial) != digest:
                partial.unlink(missing_ok=True)
                raise RuntimeError("Download thiếu byte hoặc sai SHA-256; sẽ tải lại.")
            partial.replace(target)
            return
        except (OSError, RuntimeError) as error:
            if attempt == 2:
                raise RuntimeError("Download chưa hoàn tất. Chạy lại cell để tiếp tục file .part.") from error
            print(f"Download bị gián đoạn ({type(error).__name__}); thử lại {attempt + 2}/3.", flush=True)
            time.sleep(2)


def extract_bundle(archive: Path, workspace: Path, digest: str) -> Path:
    project = workspace / "lifemate"
    marker = project / ".github_bundle_sha256"
    if marker.is_file() and marker.read_text().strip() == digest:
        return project
    if project.exists():
        raise FileExistsError("Đã có thư mục lifemate khác. Chọn WORKSPACE_DIR mới để giữ checkpoint hiện có.")
    with tempfile.TemporaryDirectory(prefix=".lifemate-extract-", dir=workspace) as temporary:
        with tarfile.open(archive, "r:gz") as source:
            for member in source:
                parts = Path(member.name).parts
                if not parts or parts[0] != "lifemate" or not (member.isfile() or member.isdir()):
                    raise ValueError("Bundle chứa đường dẫn hoặc loại file không được phép.")
                source.extract(member, temporary, filter="data")
        staged = Path(temporary) / "lifemate"
        if not (staged / "training/train_lora.py").is_file():
            raise FileNotFoundError("Bundle không có training/train_lora.py.")
        staged.rename(project)
    marker.write_text(digest + "\n")
    return project


def bootstrap_github_release(repo: str, tag: str, filename: str, digest: str, workspace: Path) -> Path:
    workspace.mkdir(parents=True, exist_ok=True)
    project = workspace / "lifemate"
    marker = project / ".github_bundle_sha256"
    if marker.is_file() and marker.read_text().strip() == digest:
        print("Bundle đã có trên máy; dùng project hiện tại:", project)
        return project
    if project.exists():
        raise FileExistsError("Thư mục lifemate đã tồn tại; sửa WORKSPACE_DIR thành một thư mục mới.")
    free_gib = shutil.disk_usage(workspace).free / 1024**3
    if free_gib < 22:
        raise RuntimeError(f"Chỉ còn {free_gib:.1f} GiB. Cần ít nhất 22 GiB trước khi tải và giải nén bundle.")
    token = os.environ.get("GH_TOKEN", "")
    try:
        asset = get_release_asset(repo, tag, filename, token.strip())
        archive = workspace / filename
        download_asset(repo, asset, archive, digest, token.strip())
    except urllib.error.HTTPError as error:
        raise RuntimeError(f"GitHub trả HTTP {error.code}; kiểm tra Release đã publish hoặc API đã hết lượt tải.") from None
    finally:
        token = None
    project = extract_bundle(archive, workspace, digest)
    archive.unlink()
    print("Bundle đã xác minh SHA-256 và giải nén:", project)
    return project


In [ ]:
PROJECT_DIR = bootstrap_github_release(
    RELEASE_REPO, RELEASE_TAG, BUNDLE_NAME, BUNDLE_SHA256, WORKSPACE_DIR
)
os.chdir(PROJECT_DIR)
print("Project:", PROJECT_DIR)


In [ ]:
import json, shutil, tarfile
from pathlib import PurePosixPath

if HANDOFF_RELEASE_TAG and not (HANDOFF_ARCHIVE.is_file() and file_sha256(HANDOFF_ARCHIVE) == HANDOFF_SHA256):
    try:
        asset = get_release_asset(RELEASE_REPO, HANDOFF_RELEASE_TAG, HANDOFF_NAME, "")
        download_asset(RELEASE_REPO, asset, HANDOFF_ARCHIVE, HANDOFF_SHA256, "")
    except (urllib.error.URLError, FileNotFoundError, RuntimeError, OSError) as error:
        print(f"Không tải được handoff từ Release {HANDOFF_RELEASE_TAG}: {type(error).__name__}: {error}")
        print("Sẽ dùng file upload tay nếu có.")
if not HANDOFF_ARCHIVE.is_file():
    raise FileNotFoundError(f"Upload {HANDOFF_NAME} lên {HANDOFF_ARCHIVE.parent} trước khi chạy cell này.")
if file_sha256(HANDOFF_ARCHIVE) != HANDOFF_SHA256:
    raise RuntimeError("SHA-256 của gói handoff không khớp; upload lại file.")

RUN_DIR = PROJECT_DIR / "lifemate_runs" / RUN_ID
FULL_DIR = RUN_DIR / "full"
BASE_CKPT = FULL_DIR / "checkpoint-1524"

# Chỉ lấy file của run này. Không ghi đè code trong gói (code cũ hơn); bỏ file augmentation vì sẽ tạo lại và kiểm hash.
prefix = f"lifemate/lifemate_runs/{RUN_ID}/"
skip = {prefix + "train_schema_augmented.jsonl"}
if not BASE_CKPT.exists():
    with tarfile.open(HANDOFF_ARCHIVE, "r:gz") as archive:
        members = []
        for member in archive.getmembers():
            path = PurePosixPath(member.name)
            if path.is_absolute() or ".." in path.parts:
                raise ValueError(f"Đường dẫn không an toàn trong handoff: {member.name}")
            if member.name.startswith(prefix) and member.name not in skip:
                if not (member.isfile() or member.isdir()):
                    raise ValueError(f"Loại file không được phép: {member.name}")
                members.append(member)
        archive.extractall(WORKSPACE_DIR, members=members, filter="data")

required = ["adapter_config.json", "adapter_model.safetensors", "optimizer.pt", "scheduler.pt",
            "rng_state.pth", "trainer_state.json"]
missing = [str(BASE_CKPT / name) for name in required if not (BASE_CKPT / name).is_file()]
if missing or not (FULL_DIR / "preflight.json").is_file():
    raise FileNotFoundError("Handoff thiếu file:\n" + "\n".join(missing + [str(FULL_DIR / "preflight.json")]))
state = json.loads((BASE_CKPT / "trainer_state.json").read_text())
if state.get("global_step") != 1524:
    raise RuntimeError(f"Checkpoint không đúng bước 1524: {state.get('global_step')}")

# Bản sao lưu nằm ngoài full/ nên không bị save_total_limit xoá.
BACKUP_CKPT = RUN_DIR / "backup_checkpoint-1524"
if not BACKUP_CKPT.exists():
    shutil.copytree(BASE_CKPT, BACKUP_CKPT)
print("Checkpoint 1524 OK; đã sao lưu tại", BACKUP_CKPT)


## 2. Áp bản code đã sửa

Thêm `--warm-start-adapter`, `--max-steps`, `--no-load-best`, log từng mẫu dev, cache ASR với fingerprint audio/model. Prompt và cách suy luận mặc định **không đổi**.

In [ ]:
import base64, gzip, hashlib, json, shutil

OVERLAY = json.loads('{"training/train_lora.py": {"old": ["5def9559c42cbddab790e2160a80a321601c9fb7027d350cbfa0d6ce4535c34d"], "new": "aceb93dd19a034c371744886b41a134111a3ab5f064d30a21bb760e8fc80792d", "data": "H4sIAAAAAAAC/707XXfbuJXv+hUoX0K1lGw5TjajlDnNpDPdnk3adJLZPqg6OJAISRxTJIcg7bg+/u+99wIgQIqiM9Pt+rQTiQQu7vcntKuKI+N819RNJTln6bEsqpqJPC9qUadFriYT+6zal6JS0n7/SRW5/XwQ6pClG/v1KOqD/VyJPCmOkx0elIhabDOhlFT2pPaRXlHCTgBk335EQJNJIncsK0TCK7ktqkSFuGxJb6ds9oZlqapXSbqt18sJg7+7tD4QqHlRyjyU+bZI0nwfB029m70KpkwopupKiqNej3+VBBbkbIVkzfEwFWZpLqdsV1QMP7E0N5tYuqMnc/ialuF0bVCkc6RBMtT/LBniFbG6uJF5+k9ZRcCdLzyT+R5JSPOaKMBFGheADbxnl+z33kL2+5gtXj5/de3hK1Il2f+KrJHfVVVRhcEn+XMDKEhm9hwbVbONhP/Vd1LmbAFiTTSYYEpwjlIpsQdhxIzYF5oHmmjzBenWtKwCuyFYrwlAKe6RVbDfY5tdtFqsV8G2yGuZ18Fan3h3SDPJPleNdJQMr7cwk+ZYosDpoAh4rFBThdqmafy9yJSMmJKgmKIuKhWHQRRELFgG06knWVWCIkuA2EphLsoyu+fbg6h5LY9lJmoZtjs6vPkdWz0EVZHJYMkCUFTQNQEIwikW12XLnxqMRALyj+uoA8yeGyPlERNJwvcylxWZGC9B88vaUNPucwSguoEaWzqmqA2eEnVO2oCC3vg7UZkM9+aAWxgcgICiug+m3Y0n+tR5i3+74EHTSXCeKQFskzxNnk0fl0xtD/IoLirUQVVfqFqWyLpdk2VMc4Xhu7SSLBiC3CHwUTNMMbGrZQUbj8Ut2C8zqL9mmyYBiPGD48JjF6pjnqF91dK9npdFGV7qFZr1QIR6UjusQkRfJ018N7UWbSlbLZFOd+h0zX4T+0igr+mLurdlxAWMSigF73QEp5huwAJbUlmr0YCG3KVfjGsw7vChPSxI87IhDEjfNX5gBKJGG0DCj0LdwDswY/bbLhVOqYNMbGSGIFazxeWlXemRB0rTcqtPubGqR+Nvj2meHpujcbhGEYzf9Tyu4dj/v7M7VT08ez35Tzs9Izvk3lc6vP8zZ/dLHR1gPJn8wSUB9F/2TjRKZB9hN1j9uyLLkE4txZagJSs2P8ltPaHHqA6cb0WWcR4qme0itpMCkxq19NIDz3hMhgGAt4dJ+/QuTSByknsNkX8GyMrT/rVWGPMGFcae5HlsXK0w1Ksb+EcrPQk/Mv9vl56B1XXOpWYFQNAIzthZ5Dr7NB4oe5kng+tR4sivudMUOIzTN1iBBmoO70Imyk4A95wBQb88C0Oz5QSIcRG02TqJk/0n/unER5Fk54CPAs9oBZLU96WM9ausAIDdOH3qzTpQjDifAtL6uM5m/XR0t3VslGdzMC8VGo2lJxXI32bh87fVvjkCrh/pTZhItYVkFFGPg7+9L354y3aQos7qBjLX+iDZ+3QnP4DRs7/dQS6Yyy/1TGxxOSuLLN3ez4Opd9IcLVeYI8JgNqsh2OQzdE4ZeANAUjRZHQdouRdg1ltwIDK5ULv6glbO9cpRkIm8/SqAsO5rwB0h985mmPX78OipukCin8+uv539OYekvdnWs6sXl/81DrBoalCaWZJWAwAzYCeUOXL2s4a8mWVFJcYBKikTAEXyh8TfAb26vHo5ulOWxfag7N4dRAV/9zhbxBc4WFcGMx0iB3F4tfhmHFAmRZWDDc7AmcszqCzk7HoUyL4SSQqfQfe2zbHJKC7MKFscxGoxzhfkOeCT3/zqzSIrD2Jw9/Orp3cnVVGCmpzhxuX88sW4WG8hEs5KWWkBD+vGOARRZffAv6IsUTSY3qGgByFd/xJI9QGC0KHIkvO0XS7GzTtVAlLNHmAAp91OHGBOJDlYoxw3myw9prVxQDqlHdaVy6+Agj7n18MQTV3MKBH6Ug8TErGDzMo4eHcoCsjNzVpGvQ1Y3giohpAQdGo6lzHVuhrnAebmWbo/1DPwg/dPnS23N9RYuXBpPrZEQFOpjYKZBLkxLDfe/fjHt197diUxZxpfDWoDn2dI8WyLmJQFcHh8z52ojqAhogK/kIgSar4gOikS7Z8m8hOuZoL95bu/s6rJWbhDfWUFRMAjEnyBBWnSZPCpq4BTLQwlbmXCKE7eSSROvR4oTDt/AZbSgiGvmaZyzj40KNPsnskv26xR6a3UzadzfHgiypKzPusML59iyp937M0llAZAqimdayg9IFfL7x1nmK7NsRek4DslB9tCYYfL8gw0k6VPyAwZOKuLWmTatIZ96FMY/3dxp9FzLFLsM1oIIHojEdHQAGPPI9bkUMDke5lMx5HLixkq+mwj1TlDfQKx/4GziTVQkNQkE6sn2DyE1AosOYVcQkJBWGCBbw2L9hR3cDC5d8ADCpqvNQRi6r+hAH+lBBBQA0XC0tg7GFuNsgKMXltw8G8FCqtA1NomKBwxCEdMh6Mn4h/IAYjcpZV6whl+OoIbWoKJyGqLHQuzVXtC5JmuKJkGpU/F5BfSXXO4nxD3u6T4dI6NIJvlnO+ZGmhS90thD1NneqZQiS3mc79XmlK/IaTTbBbD/SyGk9wijQ9mBxwTk47E3CtKO8xaYjsHtnNiu32KXotbr8VtZJ8CxYtzFKH6oCQcUhG5uIhESyZv4bBbbBupluASPEAN2uCoJSwQoiYMzr0cO9e4FQsPHE3BQgyYkCWn2ymGmjNnkOzsEbjOnUz+hZN/GaObJNk5/k0MoMjF9dwUvln0EMDwwyn8cBN+aCu9026co31wZ0znEBkKZQTrXDyAQyQg3Q8iQV/LY093TNKJDGm51dOWNnsbkxoFPwvMU/wVhJDFlNDuhk7mwOaQtedyLzxxmooc0TFlLNY7XN7K6h7CUL4P8bsbefylyM0QwLRi8uZY3qN3zcvJcIdGz5LmCIeA6YPzcj78QtfYEF6Au3zw3bZJhL8A7DI7WbQRW8iiEoWr83yeSBAqth4V6DW4KOx0nV+9AWM7HEV1Ayup72XHWaiM3HRo91XRlMp0LjsNKwjmegmtd8zrT7xAU/yFqDCXJ7MtA58eK5nJbS2xm4mwQoQ1R+W8kfeASXG3Ctr2cbCmLhU8dO1RNZ2ulv6RuqMF/IPYh1CVrEN7iOYmLaN3KwR1CpI65f2T4bUBarqwFfAg3AU/KtRImhu0xzwaJujsht6ZQ+GV3/IwB3Y11yz15416opdwTKbJ9Z+2lQFhDFReo5zCWrAcFHFnionzSx1QaAunJsd0akMIbddvDFl+Ix0KiF92BmwYPAGfd+E/+s4HLQJpnOuAEU7HBhDBtwUEUcKZRIBVjhWu9TDoOuSxrO9b7pOZAwsDUy7xrUwzrBWXZ4N7xAJVZmmNXbaHR42wGQco12RHFaNlkVOxnCQ2B0kf2+4aaacuw3DzmblCcedLf0B/XVPXojIHarC3aYD77UskemWJWK/ow7qjSCRmW6wuqeP78BWG+QjiDayCL80gRptsr0+JfNWDNm7KQ24nFUvbASe0e/vQGrg6iKsXL2Ghmf3P9QOtcHtZi7qujMnsggci75GbNt4Uwiro6Oa+JoWaH+SXJMWUMOw0Q/0YjZkEN/rhhLYFJUsBHZrqKOAo+PiHq8vrVxG7vvzmZcReLq6vI4a9LYhpV1ev4AUlddFZzXp0QvJ8JDqPMMyJ3Tky2zsZM0NglpU5zezyNmQPHAECwuDXGfC2Z0G95iLjyKTve5ysnpm6moFrDoGMNSVWcw8dDO189cmSdxcYU4wfzvJrDvi25nUnIZ/BnKKUia1128x3KEmPW8onHbswD63IaTR2DgjtNK4aLYiUbclWvfsYHfs9rwDjlu1NbcY9y+OIBzK4jjkho8xkvWjzJ/OnXjR+yrPMmzKhKTYQbKOaVoL4vKWX37zoLzYLV3jFZ47KQSja7ey3bP7NC8hQ2GLdzbFNzcepxnMEG+grEzTXc6Q7hBwkzsRxkwgk0/iwPheQo5BbKumN2rH7D8xzQa/tXXETZoBz7q3u8PMkBXd+wYJ2MY0bAgcS608o9+bHG1ga6i/KjDjlF3CZvLjpYTG/q0C6HHU39Ea6GosIJJcAjPgKx93BP/IgYieXlLxsZwwAuFeoGQ7e8Sad6bM2ck8wd1j7ec7PjchrMIuEU18wpP9yfctK1WAseHWgAXpoeKVtZ7o8zdJJ5ekel9zV9tV7KFreFfku3UcM3BLHd/qcCO8cIDv1Vw56zW82NusBZjiA8CRX8P4IEreA30JM+IAbvy8qPTJ+/yFi34Lav82TbzG46GN18WBopLIYdOR0mRuHE08g7boGXLSY21ebfEOPNcs49WiCfHcdDCxpFDCsaDYQq2n1OVAd5sZdVk/cbZaNoDtMQ1RT5s6BmcQ48BsuE2klGXU4gI4VaI79Z3Y2OYCBnrHfpuAlj6KMH4J2ukkl1Lap0Ca4XgLOLwIebqGo2qWQvHDsWfeIh/Qg5ynmMdhQEqZ3lJQiGCD5CS1x1OL6iCHf2xaNq7XRuLpIDC/iN3eo30AkAqokLKgE3XygAu6xj+Bcs3KOq7cCQLWlHhGquwCG3/DKmYPDu4p7XSPmukRxv2vkqanrBsQn/YGIbVKhQDkhm8BuolA3Rl3fvf3x09v3/P0HT2l1yoAcbkBi8Sr4GW9r/IQ3P27aT7ftp6L9tIeo0n5pyvZjUtzl+svaZ9j5nosLCice5CN8JpV3FzV0/75zFcerdU6hk4PvysJ4eZ2LkqM+8cGdFC2kM1dBFVBZTp+dWDAu/yZmYV+QPen9ilt3b00fCRJtUJQHffKz6tkarExriH3mjoGXbNPQpV3q2CpmcsTB23eVSe9atB3kHvqPr9G6qMltmPlM2VYiZiWCIesZsf7cjTw9eoqdVE+8lzbjM5KE2Ke06eN40Yt7fnfg7y0WZvZmePhwBujjyABpGpzGWAnmvTyhqBvdLBkdpTMXb4lsQtaRwvFC1xG7S7bLbSL5Uau+vvMmbmTr+HRHXBdZvQhNzln3fXVz1Sje2WD62cC090rUxEfh5K1TVJdExb2kynmXHKpp3cnQNxn0Sv3ZrWobwXGvL3zaKX7DLkkGbLZw+7GFboKUPmwj6u2B45QrHl5G7BleNdLjj58cAngJvb40wfHShPHR/iNvZcVbjTN+Wk/lPCeNegsulm7T4QT+uR8L9nsEqhFceJMiowJ42v4e5zM0Xeq/xl3uY9TxsyO73Xzg/G6viR8PtvajbuKFQzsT6kUN9UcSUx8Kd+YFb5d4egOOI91SXuD2xgEhhOO3wJeqFGiM4EI2soZP/VvX//G8gbKG3eJlN8ticWz7x3TRYvHSrd6V51efLCb/FQel2GM3OxHHO/5qg9PZXoEI/LepgffmWIBkmhzwx8o7a45Q8fUYhN1xI0b45GWH1BQaeYmCA86jK7grqhtwPXaeOTW+rWpyl9NRiTE4J+iVGucqg8+21J9MLN7+MKLFctq9V2oy7Hb3SXDSnshLqwcz3R7UubnACG4mwfwwqLDODGxKNHz58rQXNLCuc29dFko/1aeTE0Te4wj5lj5hv//JznYnU3MlNBLXHyroCQzNgTiqNYv9ugADonG1W1GKTZql9X04xTuhb2L2inrE3nK0SoDBVVOWuv+hUekpf89SqFPtYUBxoWMencGBDWX2cs5Ia+u1PdkURg8dRB4H8gLSSHt/nZyPVckfzMO/AhPNBQe3w107NeMju4scDK+vebsiYi76t7sMKU8bhSmIXWXYliXu7b+l8acIh/Yu9Wgfy+kq+12rrNOog+XJ5I7GCLrmCr3Eyup4v6txQsdQV2PSDYrmLvgiYt2Ol8MXK4unEgK7pDfqn54bti8HwjNiYi8guJWdFM9ZPfmxAZXrNQZi3Ybx4q/XpYtPG3dRp6Eb//pk1Cs8W0YqWceey+qywC6wqtELPFvzI4F48KcD7mcQ3sHmkjFiTr86iF1vuFuvttmHudHIu8P2Xt06ZHnf4YZPZv07kWVo5V0OmBs2W/MyHNzSrQ7P3BCJx66PRGMQ2vF+PHqnwPuJ2tT93PK8V/Ib3JZW+jccvlwRj1y8MF4Wr1/xk2Zvr51LqwLfJrTdaGcAOW3o4Ez78ZpWej7QW9qJJp+oD6Hvg9GFCltp4tDFbXoMpm0BB4T3Ehv/slPnltPJDQhfHbuBmWIpvuvGU3Er0gy1NjyZ2/7QgJiOdnL72V3Gop/pKajl8W4o+9PHH+fsO6pQ8TM6a7ArscF0DffjpB/TX2XnPadpHJCe4u9zcihvOcfsNeAcGcF5YH6cRVyZ/Atbv7YEjzwAAA=="}, "training/evaluate_dev.py": {"old": ["de8f185549370bed1683a83c9d60037f83800b09dc40baef6f3e3247fd798ece"], "new": "17193276070e5d2b172c824f0fb197608d39795c223c1fa5c4a6bb79cd078695", "data": "H4sIAAAAAAAC/60aa2/cuPG7fwUroIB0kWX72hTNtnuAm7N7Ke7iwE56H/a2hCxRazV6HSn5EcP/vTNDUqS0su+B5kOsJYczw3nPSIVsa8Z5MfSDFJyzsu5a2bO0ado+7cu2UQcHdk3uulQqYX//V7WNfW6VfZLjvnoYF/uyHpeHpszaXORpnx4USL1L+5uqvLakP8DPg4PLsw8X/PLi4iNb00oIPJYVcBglUqi2uhVhlAA7ounV5mR7AMQSRJSUjRKyD49jpnoZOjRHLFAyC6LoQFOtykLUaS8SZMTS1n9437YVF81tzKo2zXkGEFW7i+Fu8AuvXeEzccFVV5X9DGUN96uURXrVCZHdXIqs3TXlFyFj9hHwn+6A9dk5OTSNkCMzTSEkbxsBVxE9z0UvZF02perLjCsh8oODg1wUrG8/i0aFt2k1iBXeOmKH3wBS1W/gx3Z1wOBf08o6rYB8DgL1VJCMG2Hw/vxtEDPCEyVVeydkGNFhKcA6GviTFGWTp1UVymDzn59+5NtXcMChjllRpTu1BsBP79+9vfj2LPJ55MVJ2EmRl1kvcuI0Zru2yh3TBci71wxXouhByuXupgeOzR3H01FslxCB5rIsgJWeDrJW0jMd1/i8exCVkODWaw2jMbS3QlZpB/TUUIcgawJKsnZo+pDoRYalyVrECqBHz6A0VBadizRS4DkrFXgS2kwGwgP0ltARcKuJAOLposeWYftr9pVDBs8G2xEL3eorsxqhNPaXGZilYMfJsVELrikOtpfd+JrBv1Y1+tm4AAf1i3u9uCH94dOWdHcNRr2yihiRJTuQRtCktQgi9oc1IfXX9pRzngKLVnAaB4fIozQ/9AgCnOGH1aHGUADm+PgUxR6Z2ZZlEJU0JUDs4fJI52Xm2g7iTCmQm4l0iCzi2OgrbomuZgYCFiyBHxtuzOqIy+MRTeqzeIiZuO8EMommNbKWlL2oVehxmGb9kFa+aAhwAzi2IxBpxhIj4kRi5OT+kGwBVAWOERTDly8PgSPhORmYVaP6tMlEqAnH2oeN33nbln0HMMYCe9KCROzvazDOv0wpLoof/wmIm/baoDqLZXp676RZ+CgHYZygTu/Leqi1G5TNzvcEiqJk49Yf/JUFryBXKBsTdES+IwPZbEpSaAlhC7QiMLgz0YBhSoj7OoahaH1/xGcLPiMVbQkbeTQgGvnViqbjFOc3hydbCBQYTnSYpH28c6pUuWvsTSFE2DsA65ACQH86RIt+AytzB7f2SeZIB+k+eNnNHOV2z36mpyytPZ1nbdOXzSAmGwY6SfM8dHiiOQkjgo0D2aJFH56g/Zm7L8CMN4/22VlEyea3fc5yydwWbdI3Ssw6hjvCRok/NPlllJhMm50IUakuHUY20WY3uAu+33V7gZ2DS/bCD+/TFfrBb1J1M9O3YQ8Tv4MJH4HGis2wUxyBjeiJYsgeOC086RvBgiE6xjaNwwQ3eyeqOima2YBHKxLkb2vS5NSE+Q+0E+ZCZbLsUDHr4AxLGqQCtRSj2gyeWH8jWFHeg6P0Mi2bw1zcMirmkiDyiKCpcZtEwuDwEKumQ9m2PQRr4C4dqn4d4OKRTO+OCNerbriuoEjrhepfRqaKHgnvoYIonQmlRH4EEEcAkVDZ+TIy8Jii3Pm49Io6uk4VRMtGEJaXkaRKHpKMXgarqvrXgKV52kG6+wVcZV32hyqtu4pSYP/QiTUFInuR4xfPS4H1sn9vvaKOoKBWKD0OSVeWmfoV94fkaFg5hHgLSG9E1a2DH2VpDOhfVxfvGQqTASgjqyFwFp5eXbJe3BvfYrfKOYeO7JDqmhy7COAi+mU1ZClEnJGDC7JmyHXEADZDYMJpz2pRt6WCNEPkIThoy1eQGmWrFBPa+rGVCyb1pKHse5dxOOp63DmOJhl6FZE2K3A/FGeC0CqkJo1KE72LfRpgQXmEooF2AzLrOhj64vCvgamMl7uaUJ/fBNAgYXoEBoLtJsCtYGsOopsC+UkPpokjqxy9M2YB+aK5s23p4NSsyQu109P/PC+lKb+wMfuSohMD3NjLuXimA7dJy4DV7xSXUeoAvmaPiNEWhyvizIZDjIMGxybAEB5sTW/RUqaHpU1ADAXbJ5PMb3mZY43xKNs7EBPZIiyZk7BIOWPsXD1FQWgB+d9GkUYFoFSr4JHlo7NrYU9NMrbBCVMvcUHk4azhbuv3Z0QG0D86SJ8a7D5hOWeOeiV4ClbO/o3d6ZmUrQyDjxDBr84/kg+++1axvCUClPUovKe3aVml1+gpyCazIWbiBtrcYm1UMXO9/9j221w86j4mNn1nSbGbN5Zr06nnMODN3NKByMlHB7K2ricGaOcAii3D/gacC7YWGxydjxVC1wqgcpGkyXXSow94WohFHGclQGc8c41C4woQrUdADMuWJ+4A/GPG3LCKXTo35GXLPRh7VCuAhAY3GcchsyugpMYrOLE9dwV9bZ1w8MRchKYf1AD6dOROY7e/fwPoDZAqRjGuBw6+chzcTSrzEZCWIVY55MuIG3HnkHpyseWWgv6aZGQszURkYSQa6j9gjEODAzZja86AjUW/ZMeORox9ENRfosMgYFDaVpeyDYSYJxP0KJdxTGcUUO2gKpzZRfwCM4uU41mNjEc5RAdO/kspaY2ldGyZWpu/vvjGhnlt2AC5p9dQqWx17OTEv/ElI+vHkTBGpNV4EMOTYyrQotM98mqp1gZ+0cx0HRtG8T4HRQk5XO+jKXk1t0fHNBuG09V+i2rLibG51n2Qh8LV5RYJdgz07INR1e1DjM24B9S2iot7aLUBBiMswdt4PoqbZkvvcV7pc0HqCVZsoijag9QP4Rm2HiddE0Ub9CN3SHuuq218v3W3MGe0xI2zUwCijfkRqMS4rsTmdLwdKCKG5nPT3jV75xUoqIZSAwtLoeY4ZrsxO15k2Qp+1OMUZl+HWu8j1JN+fLKNkpAZOBRwoafByowraOAJxvbzkNLuSo9A5yPXVuZCar+BWgcqMY1kWr1gnJTt0OQhtaDmDKBiJxH7aiQBpr+3PZklEwqzvbHt95/HChS8oskeOHTEdSofQiNc5c9fiH18MgNjfcSFKawqOvAs6i5DMisF2bLJUR9B10KH9uCvUGyY/O5Tt+CNBLRYTKlE+iYyMQ5WI7+YsVy7AZxhckMHcIAws/3u9TGa0lyPiPo1mBAAvHn9HMCb13Mb6968eRb4DWKDuAIAWhfwbBWOivCMzNebuYBR0/VQVugsWBuHUI1D8/WSikz9hdUyTrg1fOQNrZQZvVNpOA2Cs3LWO+pmMJPjc+d5HgF1a95JzzVfpGqG6+uR+aMJL24MTj/H0buWpnkT4I4SF2Y8NgWG8GuMbeOF4iXWxgLcBysVxWmMzFs/oTukNkovodRHXExU/vQfZ4NTh5se9gfoeBgriwnBFyOum/Q66hv9F6l6q4RGP2OoZa/YyTPZ3R3i9AqHoq9DtJzrNSjmFmsifiGwaCMeIlF3/QP3JoW2DdH4Tl7S474Rr9eTXGKw6+HZr8c7MfE5Si95PXCKgv7VZ7pbyHS/IA+qBAFjMM7dAn/XXUI/eXtewcV99VBr8Vu0A+62j32/6CLXfLGm8hz8uaIKf3h7YwXKxwii8ZiXhEuQOlpgpKaHRZjiBFrydNe0OEgB0P/n28JZKehURCm+VYu1oq8VhDHvNekRyGFQW6oYx8xv8u7q2VrApwoBDmkFb5Ht8xOMeSmjNpE5qfxNz3wHAGmLosxK2M3a+rpsRH6oslYKtJV6qFIbM90AAYwHy0gCSIJp2VWDGYeU6vAupuM37yn94RqVUeaDiFZmZoRkymhaSbIhT5NS8ZGw/4pPT0AudWtmZyDj7Fq2VdUO+AHGz0MJhg4CePvp21P2zw+f7ES7VQk0KKVsm03w3Tn/7tM/+MX5+ffv3p9hFGDBSbAH9vHy9P3V+cXlD2eXV4vA+A7OzhtoYOn49fF8/+787IfTj2f89OqSvz19+53Ggt9ouIHUiMP7yCPyJo6/Y0hjJ3OL08zJBag3t7bNvmHHnuA1IvyzWS1Au/quk5ASw8K+coBWjRWlVD17pIoHEERP3swY42VRDeqGutrITnficTSyN1wyYyWd3bQlAFj4u5v+5cmELoWxI7cjV5tAoGTQtwWEUjfZyEMCRWah0+T41Qi+rgT9irQGIPSMibjhgJ0MAaCTtbOGKYT92iepP+elDM2nP2YaIO6h5OTtZ0+Mexw8j7jtQDfBXYDzjNmgeqxr7FxVS9OkE/cGFzUba5msTzyXBYeECIMqmg9v4ISb3HhlEok5gQwnsD/SxyP/Bb671PxNN0Qw1MfjZ/GwspTp/b/9mEA3QmWOVZafI+N58ov3c108SWtmZDylnQxdjsKwpG2Hv40WgDez/mpLDja4S+9VGlN43W3F7E9T5E46yR2+sgnpFUU+1J0KNWHUssKP3VKVleWa3sJixRj85JecM1zkpOFED75FsD+yk2Oqo9jMVnBxdP2pwkSVduo5D4KG2XjY5IyNL9aa8vWjT+3pyI8zhoAV2frRLKySk+JpIfI8Y1yeGLKqVWL8KIwy2XqxCbTvtvqbiedpILf5G32ajpBa9XuliW4R85JudQhs+vXXVsvP+bkW7m9FiiMLkBzn+EKHc/pyhnOsCTg3H8/oAuHgf+gwsnZsKQAA"}, "src/lifemate/runner.py": {"old": ["bced4f1465e248dee7167922dad7bd8b68ea04448e0d8b226b50e611c89cb7c3"], "new": "eeabbfbb3b702f7da79119230eb155fed46dc2a8fdfdde178b5a08c64f2629f9", "data": "H4sIAAAAAAAC/8VZ3XPjthF/11+B8iVkI7N3k5eOMurMtTlPOtNeMr5L86B6OBAJSoj5VQC0T3H8v3d3AZAgJfnky3XqB4sEdheLxW8/sCxVW7MsK3vTK5FlTNZdqwzjTdMabmTb6MXCjf2i28Y/t9o/Kd4Ube3f9GGYMLIWixLFd9zsK7n1sn+EVzthDp1sdn78TXNY2PG04Ib7YfuTmbatMtHcL1nV8iLLgaJqd0umBLyhahU+67a6FxnvC9lmuOw4prtKGie+bgtRab/A+06IfH8j8nbXyF+FWrIPsNabnWg8fQf/OzMwbHtZFVkttOY7AfZZFKJkWpisEEaoWjZSG5lnWogixn8rJhuTsKu/sHdtI1YLBn9OVNPX3YFxzZouHDatyvcLGrH2TQdhCY02XXp6gjjTmjc9r7KTc3lf8JAg41V1RLTl+Z1oCo3UTZNONsbW7IPqxXnqrWjyfc3VHVBe80oLZyJ3LrLpeqNjzeuuEitWyNxstAGzAwBul4wOKiukWhFSEmuvvFcKDgQkWr5NRMKiW5rdg2KtOsDsht5JtV41Uyr8K1tFM3AkTlK6EyaOnIRoyTa3yUAtSyK2JKqtRJSw9ZpFvRYqAicpgmm7kuW16ylRalTJ6X7LvmYVLBMrcS+UhqNxiyaJIyfFkAu0OMYy8JWBeRLaC4zhVpDp1pvZADR0ruQWUE9bnNj6yMJLwB88zt3AGd6qg+vjXk4dYaiTY3EyCuAA2WmgEADvEJM0Bzc31Zlse8hwN2vSMv5VdrFdexQ3NdQR62YGDQDTEY230T2vJEA/xxAX2585FNlv5K/kuFuICKtwcRyIB5xYAeMrAMMOOWjQs8NODv4WTUillo02vMlFHHI1vAa8gXWNSi6h52rX14AzjUy4FcuV+C034qMZdozRbckonvpTHG21ZEN4HQfpJHRf4UMJ4QKdfk0Gsobx8RAOcBogRzO5lUYXHVccxvzKJ4j0OtRmIOiUbFWGZtVr2BAaw2YLGoqTOaXbxdrvZowNflf+YekMGJw7GQ4SgiCQ+i0mE1hxI7ydycD2GZxItaWsvL3gBJUhF8FMmXZClVne9g1E2tgu6YQBLxDhzrx0d+wbC5Fbv8ImgIALYhDA3KqAGgY5PUhB1iI0uYnIYBo8DcJ4BJHqpFbsymsdmmRUc4jzpE4G8SnTRnS/F20vtZtdHqPVF4P8C63ZtZXMDy+1J4hHmadiU7IKc9KzSlgbtV1HNqI45WQ8H5eQJ5pIGbYDuR28hkORBVvxpKiIXweqKcGijkNWy0ouK6glR1EOJqjlMawv8xhvnpHVbqC9iwLLeBcdg/HFB6bzvag5bNIoKTTt0s3ZhWbzS/YqgWz+ehGGDeDBOjQtoKbT8WOkxC8ih7OlOBSthk15YUKpVsH4fFN2PHlCrGqsyrnOpVxTJZVc6AJ5CyVHbmkmXjAmki+UAfxDECr/H24ybPiFnnIW43jUhwyYMp5NU/Ycz/Jc3B90GoH8DGwH6pfZ8Mg3PVId+HgJNiTgHqLFXH0Xr1XfOMV1/EJATFLq0qOq5h8p8h8FbyxZMypYebMT8UAXmGOM3vM0Mjm4z8PvTF2n3CB4Uvk7RcD8x7CxRhwLm086pGXCZV0uR0xdnsdfDOfPyedHJVLKIbo3RWxfk98HzOGsIwc72ZSATGC2BxsWh8Nl387Q9YIuDXZgUiPCBcM9ECJsLQprWQsQCAPco1RCFMBCwhs4ro2slshpT3XtcnFxxQFCcdZpHQcKp/YZD3g5XF1lgYet2tasQ1JUDQdfZOi072xJqJU/7/Wr9BVgexJc7ViIChzxd7BDIHbPIYlfBujQ3wI3Q989dwddsmPjwMWR7qCfCfVg54S000Cf7OuMzwxbsHKG14F8EiknQl4Yhza3k1gZYPVMTBIfc9EZ9pZ+MCxxzahQCAEhGxOXkdVi/ejQ9pUsvrp9ssTrR3PoREzPSZoRLLPsacUeaegJChuCvz7A2ZgCBpMvkk3LSHjFV6dV8LnJRcj45EWO+VE4Rsi1cKL7OHnxRcfw6gK8nLjoxH+0+pEeCS55NqbYitiS+yxLfUsAEK4XY4fTdSjDCAFalXJH3pBRl3LJqqr2j7zgHWbzbuyL+cbMpM15WnoyCa3AM+usxjOfTMJYC+Rh3/U0LWqdlzurMzwAk93QJrIt1+iW3BWj3/EEMLlGHZAA67whFdSu3joMygm3KPC3gBPaNcgfaLfc5PtMg4C1X7KStTRel2wkCNms82QKcHaKj7pgAY1nTcZMhG1S30ceVR9OE1V3ZjqnenjcVJBM3keUW7DNDequeAFPFLRCaok183xfGItg1OCFwbR3cP8Ij2qk23NVDIQ0DIVK0MM5KbgRD6PQ486Kze9j6h8jJ2Z4a1PvSg9Kgv+7LgwFY+pIOsi3vRlfLhHrt+jzvRBYf2qwnw++nb9M6b6OXwdl7PD1IdDAhSSrBvBgkzUOlAqnU7g0Y0epvgMviu2LXmN/HaLMR6lN1t7Rq2V6kHD0jrOF2iyOHiIMR3lbyGa3jnpTXv0ZLvSQHbQB5erVpPNtEXte8Wk0xwt+ULFQyxxMMyWy3ygkAPBfvOrFW4zrkIS+6yE0gKnFsGYBWWaSkqJk2rMA0Skvijhcc0ZyUVVioxZkDGyd+jyB4W5ab04E+8Q9oMTB5hRYjrhPw33M6aFPT28ZvluucE+PuOPVrEiMxgQIc25b0bgvZBhenmbWQgCk5Ctx0JmwC57sLrCvWfTvZn4wVk5Z9ZhuZ/nelhyQXXOhtSjmVYevNR+fTbKr9JvybO0BKKwA6AiPhP1hPThjcIEm+N3YzOoB+LNqAXyPA+sT1NsPmrzg0Yt4YoR9Z3Id+TYuVnmEFoE9dlpnyKhob8V+s19OgusJZliaWwbefYbDAubo09dikttCNvcpAkBiLyaTTPIpwjAFfJK2qtqHLO/6FXUO/Qc8uAqd/HwZfKd0XZngG6PUGb/nsuJbqP5t8xEpxiWeO8Dobz999waTnBL/6SUEevvlbi/YN6+uLKqYBhzj92NXU30LTqs1u7qiFa5gBQbh7UCMlGBZIbb9bgcsqYN4S5cuqfDO/f119v1Pf81+uL7+x9/fvbV31tfREdmHmzfv3l//cPPPtzfvTxKf+RDsgwSEEKGo24eBAqd8lHPbwNvTFy4Vx0rxE4kQC0m7qj3TITNgP3Ra8bE/QVX/OCr0lFI2sTb4nyXoxX8BHXvZX7AgAAA="}, "src/lifemate/models.py": {"old": ["00b8e04764f7f7660e330e9a827b1cb87ee938403a047e5eb2d260109006ffac"], "new": "b33fba3a1f02f488324e364a1de26cd671bc66facbd845438a43142f2fe83a55", "data": "H4sIAAAAAAAC/81aW3PbuBV+169A2T6QWYVru9vMjlrtjifrbNPJZSfJtA+KykAiKLGmAJYgbate//eecwCSAEklcduHZsYxics5H4Bz+Q7hrFIHliRZUzeVSBKWH0pV1YxLqWpe50rq2cy27bneF/mmff2HVrJ9Vrp9qsQsQ5FZxbc0vZX4wjaY7pLXKKvt/AVeTUd9LHO5a9sv5XE2m6UiY0mtqu0+SaFfhPQ8Z40Wya5sFmyjVBEtZgz+VQIWIhmNiLNC8fr8GcuzdiwThRZu7+8vWgVcV8lBpaJIMkAgqrLKZR2aFoS7YLqu2K+ENWJPf2Bpvq1X0DZHmGurXqmaLWmMMzWivq2SWb5LJD8IDWPCwDTEuJHBnAVlJcpKbYXWqkoGfbW6FjL/l/A7XLko8v6BGjJVMdTCcukpNRDxH4KC8YT2WxrbdcFeYW+ca9iHQoRRP8tRtsI5axBhjSLWe37xh2chTa0ET5PNsRY6jKJ4L+7SfCd0HRq4tyLf7WsSjphhsbVIQx8aroAeYAUIMs5rUaV5ZUVM4gSbTZkBoJssy+/YcsmCWPNM1EKCGh0wK9YbsMnbjYxcC7rvNNlz0sGiXf287zOrwb6V0Y0bM7dawIPCCH4lGo4u8lbl7sLayHsAS9wWXGv2vhRiu38ntmpHh26OgKw0yWVeJ0moRZHN2bR1ztmG1+AsqHUBytAiv5szzQ9lIZKK113r+bOzszPnhK3bkX90jcYvKy41LOAgqs6lL5tavUYEL1RlIL8X/7yAnzl1/dIa86wThahjkg7KfS3UlYqbfCugL9g2KQ/wkI2v4iseNb/hecE3dN7kysG2bIKhXcNWjLyv09H5GAz0YMa40AS8ENaaS7DJkvayUFtemINKlCyOyw9VI3qZFJFA1FSA8tbULmoAh3BaKJObOYLl+aPB6Ohe0v8I+zaBvUkO4pA0mu8E4T6xnE5kBKcTOrCjWNzwIhxg7s0LgPcv/iDH3PBI+rdu2G/Z2xJTAi8YoFS5poTDVMZu96oQxua2Vb4RyYHLIxw5QC90zN5mENAlrECzVy9fXL2+/HCVXL5/lzy/fP7nK5ZrAFDPHTVaofPwpqiZ3qpKpGD+maiEhHNJFYQhSHYUyZiJSmAbvICIY/zLavVXl2z5di8Sa3BKx0Le5JWS8U7UYTBGFUQo9Y2SYlLOQdQc5JzqXzjJpsg1Pa3XfcS3AXGIzA/dE/ruvQEU0GjJEM4ohZH7mJB2YqSbLGHWiSxKciZEkK0GFLtCeo4g1zmmgl3924SA3vhgaP8CUgoudw2YPbQHN3ngz33w3mDnaLXD7YtOJUHazZoMaUk8KAYqkerwhBTKh7W4q0OwOJXCviyDps6efh9E0UguHiOJNpaEBwW2A9FjeHxjSINDpkDYS4IUetDAJe4fxkoxlE7LM+cXgB0zIxRVpxysJ82zDJOBSQ978LqqkX9k+U6qCilcDYPNFNxCUJwVjd7b6NknNLOga3G0KQ23TC+MleN+riHL8btEituESJCm3EUEDKymB82bNFcJOjHRitW6z19O2jXSvZUacjKmMv4u3eYgwPEJVQoZBtUGzoZrRCL4YbyDqHq7b+Q16kYOExb8sEn5ws4g0wjPzy6+Y08Y/gL73wRBNH0WBmnclLD/IiS5Pkh3D2JeAsQ0tJMcFhY56fKIhtsacdocSu2nl/vA33zwJr8BPM1oNbsG/S6Ih/kgApUcHBmo2DIM5shrF4ETFiDZoH+IsPWOrsdyshHZJPg+xewtS/MbYcyLTItMBgNsv7k1r8A1Wr4+ctzBuBjAC1nHh2tkouZF26wq7sBeE3U9IAfgciUyHTMfbSgx1DO0TZaIfgM8tD6UgT8zvq3AZkzkcA7o3gSFxSgiwH4aJ/e6HgCd1Fjdcb3N8+ULDs4eYeMgFvm6oRgp+FZYoO6+UjijU3Zc1quORoRSAvAjOoosh11aNTLFCEF+lPmsEwCXx1hDTAGO0FWYNiWUqjjOfP+fs5ZuZMa1MLGY5GOI2jKwVR8YHy9u+VEnF+ng0EgSyKDf8UFwGXI43uW5V3yQot8sR1zHd90KKQ3IaqvfcDjcIJ5wYyzP3JWG/QIBFWyoIE9qG1Ih1SGX2BTFXBMLlWVb43rIkewUEL1IINESbIHBuc6wuhC2JwbaM8x8QEGBdP+VF424qipVhVnwUgJDzFOLGlPDgt3jnj+MHZjG9KY0JHiPyQBYvVx8T07dMaLFcJkT4d4icfLDF4kTJCfm519MVyR7CC0aEgucC7Hf5XKj2G4h4TLcILSCuWtfIIYC3WFxtq+RNCX9GlRDFRigUO7EICeERrNTkGny6KgfDbEPMFOTHk0OJoyCrI/IgQHnBDKKEWOmsHZWpaF8aHWj3DHJgHof4hN+sgDuKcKzee9mOpoPS6eBZ1FHG3X0yshaWJnfDCevfTuTZdPbRVfghiNLIwH2SwCEfopAy3FIKnlqMkOhJKbVYH7CZhNe15AFIbCBA+trmwxtn/3qsgzKoQDfljLB8eMj4jcLielX0rYPStI2kvfVc2xKBz+GesB60USF/c5g5LSDyUBtfd7gR8N3DYw92HgY/A2YcAmlY/+ZIYXgiMGojYGyl88m9I+Q+w3DAn3MUvuPLHFX5lJNNlXG7ITEREKVjLOjbXM4SUTbk5lP9g5swn+dntIWa8txqdZzMJAV9KHhxLAUmCiZsuE504MgeyYboN+Q06cH+EFlOaC8oynD2EihIoawiATc90nrw6kgotvtPvjkdV4mYDnbnBetXuIlo8i6woAbQwjKy9DELGyguGM0r+0nbaCoGlifIR6qqcEBiKZNfL0G5tabOIlbMjOjVTSzvhH8qVaqSPCbyA8BKsXRDqU2cw1ECDLgbe6EOTuPVufrruvbYd/Z2lNoxVUCmPImrIK/f/r0KfxxgWw4+vGjfvIr/EDT77CWgB8cjoUm3+klzHn585u3766eX76/stKqY4/0BkmMX7/jdDNS3G1Fae444r+8f/vmJzovcvFewsGGbEQnOPgbAPx4Hz/5+DCG8tPbD5evXnk8zEyfDC72pL3PQR74EwsgkfGuUk0ZnjmW81XLmVJseVSucwmZCAJJSFrnZD5OOJmYRwNtvCUTNJ8xAl2rMhjNvG8HLeyIhy/JQaNx5Jjv7lCsADOWlAydidgJlaYnqx0ZeIfirNRIJH/BqwWnp5s72ocTKyKsYKKEY2GxOhAWPfBu3cM991cyBWt6dafO6jP4SNDKvK0HQG1f37R+mA0twN6ooU2CbzcaIhqllbCrFAEUpa/2hRK4gfhV1w3PSeqr1zNX99SA01/P6ct5712jL+hd14kv6U7/xBf1vtssFJJfuQw4IKRbjeGlgLnMwM2b26uo9i7oAwTIS8gT9cQlkPepuNd44lao58EpL2tRjUbYJOAIgry3VRJDmcuv/QED8m3x/zd3Sh/ay87/r4uj7g7W3tZ0MP+zi6O27OzExiV+Jsa3JE+n88KpCS1567uE0qbrf35RdcKtPSGtR7uL9azOLwDoLwJEVnd/DgDP5Miz8epbGN2Y0e73wyhMhq5iiIZgDDQSjcF+DJtY6OS9F5q7qU2MwQOMsYuwpwO3GMvo+6wEZ3Dn5bAiDNztTS+wRwg9bQHasze6CfoSo+t2GYnooaw7d8sgjrSi+802a8xThOeOCNuH+cDc5tP7E01K7J67UsZWcYOCZlQIOYUNrEwnRX4twk5aP/FxVZChuRbbV5VAncpxJfDI2mdQZ0wYyHjOFyucz1Y3boxZnooZ0Dee6fY+bib+7Q19xhlkRr908pyiP5TV2dyxGL3npVg9PV+zxXpIaLxyZwDQ1lvuvcVnCq5/A4j523sVJQAA"}, "scripts/run_inference.py": {"old": ["ace14763b9573e6630e191e39d58d6844c8a5ddd4485a1100b817a925c7db86b"], "new": "79c53e59a37bfa0251b77f7f7734f0d23f40b1efc2ff35cbda7caaf163afcefd", "data": "H4sIAAAAAAAC/41UTW/bMAy961cIOjlY42y3YYALBEWGbWibIO2tKATFphMNsmSIcrv9+1FSnDo7NPXF5td7lPmo1ruOS9kOYfAgJddd73zgyloXVNDOImOjz+975RFG+zc6O37jX2RthOpVOBi9G3E2ZDK2XW3WcrteP/IqeQoi1IboZqUHdOYFillJ2GADPn15ZoRWRqBSWwQfis9XHIMv3mAWXKCvxWzGMqvRLXQqQOkHa8GP7GRJbVsg4BoYYw20vFPaFjM+v+b3zsI3xulJx/LU3HjEcun3Q0ftbFKkaABrr/v4PyqxHSwPB+CubY22wG+J/I7I+YmK7xRCjJViNiEoVdNIdUQuxHzeqKDEFae21GBCJaK98Op1ETx1+akfdkbXMgCG93GwNzpMgT5c6YbQD2elNBB6L3DYdRqRDlzGOZv3YWpnW72fwmQPLk5/IqK8D6LQzzvXwAUuY7qPpKlG9QH8hSRj3Ou87gdqXdV5vBgcbULwA1xut1b1Aaj2AKavxDoJRBn+62F9z6PCSScq8A46pxGQLx+2nEZrs5iQKL1DjDLFIxcRIOnwSJlekRSLHNZtyiiJWyburN8UypJ3ePI4LMG+aO/sk7j9+X11t3xcSWpB3ixvfqzEM/HErUoLeY46WcvMm6dJBXGKpXGqwUldjsYi1ZDm/oSClsA12u4rMYR2/lUcYc72sTg1mkCi9q/OXUnW//myYN+cmXuSRGdI6qhOR0rmWwbpZ5pxMicYWToyXkFHmOyZpEThSBLOMT6aOYOuJUajktKqLl6qVcWFlPHmkVLkkeVriP0DzDetV38FAAA="}}')

def _sha(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

for rel, spec in OVERLAY.items():
    target = PROJECT_DIR / rel
    current = _sha(target.read_bytes())
    if current == spec["new"]:
        print("đã áp dụng:", rel)
        continue
    if current not in spec["old"]:
        raise RuntimeError(f"{rel} không phải bản gốc đã biết (sha {current[:12]}); dừng để tránh ghi đè nhầm.")
    new_bytes = gzip.decompress(base64.b64decode(spec["data"]))
    if _sha(new_bytes) != spec["new"]:
        raise RuntimeError(f"Dữ liệu nhúng của {rel} bị hỏng.")
    backup = target.with_name(target.name + ".orig")
    if not backup.exists():
        shutil.copy2(target, backup)
    target.write_bytes(new_bytes)
    print("đã cập nhật:", rel, "(bản gốc:", backup.name + ")")


## 3. Đường dẫn và kiểm tra GPU / dữ liệu

In [ ]:
import hashlib, json, os, random, shutil, subprocess, sys, time

# Giữ thứ tự tool theo bảng chữ cái như baseline (không xếp theo transcript ASR).
os.environ["LIFEMATE_TOOL_RETRIEVAL"] = "off"

DATA_ROOT = PROJECT_DIR / "data/raw/train+public_test"
MODEL_PATH = PROJECT_DIR / "models/Qwen3-4B-Instruct-2507"
WARM_DIR = RUN_DIR / "warm_from_1524"
ASR_CACHE = RUN_DIR / "asr_cache.json"   # chỉ memoize transcript; không đổi kết quả
PUBLIC_DIR = RUN_DIR / "public"

def run(cmd):
    print("$", " ".join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), cwd=PROJECT_DIR, check=True)

def latest_checkpoint(folder):
    required = ("adapter_config.json", "adapter_model.safetensors", "optimizer.pt",
                "scheduler.pt", "rng_state.pth", "trainer_state.json")
    found = []
    for path in folder.glob("checkpoint-*"):
        if not path.is_dir():
            continue
        try:
            step = int(path.name.rsplit("-", 1)[1])
            missing = [name for name in required if not (path / name).is_file() or (path / name).stat().st_size == 0]
            if missing:
                print(f"Bỏ checkpoint thiếu file {path.name}: {missing}", flush=True)
                continue
            state = json.loads((path / "trainer_state.json").read_text(encoding="utf-8"))
            if state.get("global_step") != step:
                print(f"Bỏ checkpoint step không khớp {path.name}: global_step={state.get('global_step')}", flush=True)
                continue
            found.append((step, path))
        except (ValueError, OSError, json.JSONDecodeError) as error:
            print(f"Bỏ checkpoint lỗi {path.name}: {type(error).__name__}: {error}", flush=True)
    return max(found, default=(None, None), key=lambda item: item[0])[1]

print("Run:", RUN_DIR)


In [ ]:
subprocess.run(["nvidia-smi"], check=True)
free_gib = shutil.disk_usage(PROJECT_DIR).free / 1024**3
print(f"Free disk: {free_gib:.1f} GiB")
if free_gib < 18:
    raise RuntimeError("Ổ đĩa còn dưới 18 GiB; cần dọn hoặc thuê instance disk lớn hơn.")

try:
    import torch
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch==2.9.1", "--index-url", "https://download.pytorch.org/whl/cu130"])
    import torch
if not torch.cuda.is_available():
    raise RuntimeError("Kernel không thấy CUDA GPU. Chọn kernel Python của /venv/main.")
props = torch.cuda.get_device_properties(0)
print(f"GPU={props.name}; VRAM={props.total_memory/1024**3:.1f} GiB")
if props.total_memory < 23 * 1024**3:
    raise RuntimeError("Cần GPU 24 GB VRAM trở lên.")

if not (DATA_ROOT / "train/train.jsonl").is_file():
    dataset_archive = PROJECT_DIR / "data/archives/train+public_test.zip"
    if not dataset_archive.is_file():
        raise FileNotFoundError(dataset_archive)
    (PROJECT_DIR / "data/raw").mkdir(parents=True, exist_ok=True)
    subprocess.run(["unzip", "-n", "-q", str(dataset_archive), "train+public_test/*", "-d", str(PROJECT_DIR / "data/raw")], check=True)

for path in (DATA_ROOT / "train/train.jsonl", DATA_ROOT / "train/train_tools_catalog.json",
             DATA_ROOT / "public_test/public_test.jsonl", BASE_CKPT / "adapter_model.safetensors"):
    if not path.is_file():
        raise FileNotFoundError(path)
print("GPU, dataset và checkpoint: OK")


## 4. Cài thư viện và tái tạo dữ liệu train/dev

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-train.txt"], check=True)
subprocess.run([sys.executable, "scripts/download_models.py"], check=True)
subprocess.run([
    sys.executable, "training/prepare_sft.py",
    "--data-root", str(DATA_ROOT),
    "--output-dir", "data/processed/sft",
    "--seed", str(SEED),
], cwd=PROJECT_DIR, check=True)
manifest = json.loads((PROJECT_DIR / "data/processed/sft/manifest.json").read_text())
print(json.dumps(manifest, ensure_ascii=False, indent=2))


In [ ]:
TRAIN_JSONL = PROJECT_DIR / "data/processed/sft/train.jsonl"
DEV_JSONL = PROJECT_DIR / "data/processed/sft/dev.jsonl"
AUG_TRAIN_JSONL = RUN_DIR / "train_schema_augmented.jsonl"
if SCHEMA_PERMUTATIONS:
    count = 0
    with TRAIN_JSONL.open(encoding="utf-8") as src, AUG_TRAIN_JSONL.open("w", encoding="utf-8") as dst:
        for line in src:
            row = json.loads(line)
            dst.write(json.dumps(row, ensure_ascii=False) + "\n")
            messages = [dict(m) for m in row["messages"]]
            payload = json.loads(messages[1]["content"])
            tools = payload.get("available_tools", [])
            if len(tools) > 1:
                key = f"{SEED}:{row['sample_id']}".encode()
                rng = random.Random(int(hashlib.sha256(key).hexdigest()[:16], 16))
                shuffled = list(tools)
                rng.shuffle(shuffled)
                if shuffled == tools:
                    shuffled = shuffled[1:] + shuffled[:1]
                payload["available_tools"] = shuffled
                messages[1]["content"] = json.dumps(payload, ensure_ascii=False, separators=(",", ":"))
                augmented = dict(row)
                augmented["messages"] = messages
                dst.write(json.dumps(augmented, ensure_ascii=False) + "\n")
                count += 1
    TRAIN_FOR_RUN = AUG_TRAIN_JSONL
    print(f"Schema-order variants: {count}; training rows: {sum(1 for _ in AUG_TRAIN_JSONL.open())}")
else:
    TRAIN_FOR_RUN = TRAIN_JSONL


## 5. Preflight và kiểm hash dữ liệu

In [ ]:
common = [
    "--train-jsonl", TRAIN_FOR_RUN, "--dev-jsonl", DEV_JSONL,
    "--model-path", MODEL_PATH,
    "--gradient-accumulation-steps", GRAD_ACCUMULATION,
    "--lora-rank", LORA_RANK, "--lora-alpha", LORA_ALPHA, "--lora-dropout", LORA_DROPOUT,
    "--evals-per-epoch", EVALS_PER_EPOCH,
]
preflight = RUN_DIR / "resume_preflight.json"
run([sys.executable, "training/train_lora.py", *common, "--seed", SEED,
     "--max-sequence-length", MAX_SEQUENCE_LENGTH, "--learning-rate", LR_ORIG,
     "--preflight-only", "--auto-context", "--preflight-report", preflight,
     "--output-dir", RUN_DIR / "preflight"])
selected_context = json.loads(preflight.read_text())["selected_context"]
print("Preflight context:", selected_context)

# Dữ liệu train/dev phải trùng SHA-256 với run gốc, nếu không thì không phải tiếp tục đúng run.
original = json.loads((FULL_DIR / "preflight.json").read_text())
current = json.loads(preflight.read_text())
for split in ("train", "dev"):
    if current["splits"][split]["data_sha256"] != original["splits"][split]["data_sha256"]:
        raise RuntimeError(f"Dữ liệu {split} khác run gốc. Dừng trước khi nạp checkpoint.")
print("Train/dev SHA-256 khớp run gốc.")


## 6. Smoke test warm-start (vài phút)

Chạy trước các bước dài để lỗi ở đường train nổ sớm. Chỉ áp dụng khi `MODE="warmstart"`.

In [ ]:
if MODE == "warmstart" and SMOKE_TEST:
    smoke_dir = RUN_DIR / "smoke_warm"
    if smoke_dir.exists():
        raise FileExistsError(f"Thư mục smoke đã tồn tại; giữ nguyên để kiểm tra: {smoke_dir}")
    smoke_command = [sys.executable, "training/train_lora.py", *common,
        "--seed", WARM_SEED,
        "--max-sequence-length", selected_context,
        "--learning-rate", WARM_LR,
        "--epochs", 1,
        "--max-steps", 2,
        "--limit-train-samples", 8,
        "--limit-dev-samples", 4,
        "--evals-per-epoch", 1,
        "--disable-early-stopping",
        "--no-load-best",
        "--save-total-limit", 1,
        "--output-dir", smoke_dir,
        "--warm-start-adapter", BASE_CKPT,
    ]
    try:
        run(smoke_command)
    except Exception as error:
        raise RuntimeError(f"Smoke warm-start lỗi; giữ output để kiểm tra tại {smoke_dir}") from error
    if not (smoke_dir / "final/adapter_config.json").is_file():
        raise RuntimeError(f"Smoke không tạo adapter; giữ output để kiểm tra tại {smoke_dir}")
    shutil.rmtree(smoke_dir)
    print("Smoke warm-start 2 bước đạt; đã xóa output tạm.")
else:
    print("Bỏ smoke warm-start (MODE khác warmstart hoặc SMOKE_TEST=False).")


## 7. Đo dev baseline 1524 trước train

Báo cáo và log từng mẫu được tạo trước khi train; cache ASR này dùng lại cho candidate.

In [ ]:
def dev_eval(tag, adapter):
    report = RUN_DIR / f"dev_{tag}.json"
    weights = Path(adapter) / "adapter_model.safetensors"
    if report.is_file() and report.stat().st_mtime > weights.stat().st_mtime:
        print(f"Dùng lại báo cáo dev mới hơn adapter: {report.name}")
    else:
        run([sys.executable, "training/evaluate_dev.py",
             "--data-root", DATA_ROOT, "--adapter", adapter, "--limit-samples", 0,
             "--report", report, "--per-sample-log", RUN_DIR / f"dev_{tag}_samples.jsonl",
             "--asr-cache", ASR_CACHE])
    return json.loads(report.read_text())

BASE_REPORT = dev_eval("ckpt1524", BASE_CKPT) if EVAL_BASELINE else None
print("Baseline dev report:", RUN_DIR / "dev_ckpt1524.json" if BASE_REPORT else "không đo")


## 8. Train (theo `MODE`)

Run nằm riêng nên checkpoint 1524 không bị ghi đè. Nếu warm-start bị ngắt, chỉ resume từ checkpoint đầy đủ mới nhất.

In [ ]:
CAND_ADAPTER = None
CAND_DIR = None

if MODE == "continue":
    CAND_DIR = FULL_DIR
    if (FULL_DIR / "final/adapter_config.json").is_file():
        print("full/final đã có; bỏ qua train.")
    else:
        resume_from = latest_checkpoint(FULL_DIR)
        if resume_from is None:
            raise FileNotFoundError(f"Không có checkpoint đầy đủ để resume trong {FULL_DIR}")
        run([sys.executable, "training/train_lora.py", *common, "--seed", SEED,
             "--max-sequence-length", selected_context, "--learning-rate", LR_ORIG,
             "--early-stopping-patience", EARLY_STOPPING_PATIENCE,
             "--early-stopping-threshold", EARLY_STOPPING_THRESHOLD,
             "--epochs", MAX_EPOCHS, "--save-total-limit", 5,
             "--output-dir", FULL_DIR, "--resume-from-checkpoint", resume_from])
    CAND_ADAPTER = FULL_DIR / "final"

elif MODE == "warmstart":
    CAND_DIR = WARM_DIR
    if (WARM_DIR / "final/adapter_config.json").is_file():
        print("warm_from_1524/final đã có; bỏ qua train.")
    else:
        interrupted = latest_checkpoint(WARM_DIR)
        partial_checkpoints = list(WARM_DIR.glob("checkpoint-*")) if WARM_DIR.exists() else []
        if interrupted is None and partial_checkpoints:
            raise FileNotFoundError(
                f"Có checkpoint warm-start nhưng không có bản đầy đủ để resume trong {WARM_DIR}; "
                "giữ nguyên thư mục để kiểm tra, không khởi tạo run mới đè lên."
            )
        # Run warm-start bị ngắt giữa chừng thì tiếp tục đúng run đó; nếu không, khởi tạo từ adapter 1524.
        start = ["--resume-from-checkpoint", interrupted] if interrupted else ["--warm-start-adapter", BASE_CKPT]
        run([sys.executable, "training/train_lora.py", *common, "--seed", WARM_SEED,
             "--max-sequence-length", selected_context, "--learning-rate", WARM_LR,
             "--epochs", MAX_EPOCHS, "--max-steps", WARM_MAX_STEPS,
             "--disable-early-stopping", "--no-load-best", "--save-total-limit", 4,
             "--output-dir", WARM_DIR, *start])
    CAND_ADAPTER = WARM_DIR / "final"

if CAND_ADAPTER is not None and not (CAND_ADAPTER / "adapter_config.json").is_file():
    raise FileNotFoundError(f"Train kết thúc nhưng không thấy {CAND_ADAPTER}")
print("Adapter ứng viên:", CAND_ADAPTER)


## 9. Đo dev ứng viên và chọn đích public

`PUBLIC_CHOICE` chọn baseline/candidate một cách tường minh. Call-F1 chỉ là diagnostic.

In [ ]:
CAND_REPORT = dev_eval(f"cand_{MODE}", CAND_ADAPTER) if (CAND_ADAPTER and MODE != "eval_only") else None


In [ ]:
def row(name, r):
    if r is None:
        return f"{name:<22} (không đo)"
    return (f"{name:<22} state_match={r['changed_app_state_match_rate']:.4f} ({r['state_match_count']}/{r['samples']})  "
            f"call_F1_diag={r['tool_call_f1_diagnostic']:.4f}  empty_pred={r['empty_prediction_samples']} "
            f"(gold rỗng={r['empty_gold_samples']})  end={r['end_reason_counts']}")

print(row("1524 (baseline)", BASE_REPORT))
print(row(f"ứng viên [{MODE}]", CAND_REPORT))
print("\nCảnh báo: dev chỉ có 154 mẫu (sai số xấp xỉ ±4 điểm phần trăm).")
print("Call-F1 là diagnostic tự tính, chưa chứng minh trùng công thức BTC; không dùng riêng nó để kết luận kỹ thuật nào tăng điểm.")
SELECTED = PUBLIC_CHOICE
if SELECTED == "candidate" and CAND_ADAPTER is None:
    raise RuntimeError("Không có adapter ứng viên; với MODE=eval_only hãy chọn PUBLIC_CHOICE='baseline'.")
SELECTED_ADAPTER = CAND_ADAPTER if SELECTED == "candidate" else BASE_CKPT
print("\nAdapter được chọn cho public:", SELECTED, "->", SELECTED_ADAPTER)

PUBLIC_BLOCK_REASON = None
if (SELECTED == "candidate" and BASE_REPORT is not None and CAND_REPORT is not None
        and SKIP_PUBLIC_IF_WORSE_BY is not None):
    baseline_match = BASE_REPORT["changed_app_state_match_rate"]
    candidate_match = CAND_REPORT["changed_app_state_match_rate"]
    if candidate_match < baseline_match - SKIP_PUBLIC_IF_WORSE_BY:
        PUBLIC_BLOCK_REASON = (
            f"Candidate state-match {candidate_match:.4f} thấp hơn baseline {baseline_match:.4f} "
            f"quá ngưỡng {SKIP_PUBLIC_IF_WORSE_BY:.4f}; bỏ public inference. "
            "Có thể đặt SKIP_PUBLIC_IF_WORSE_BY=None hoặc PUBLIC_CHOICE='baseline' nếu muốn tiếp tục."
        )
        print("CHẶN PUBLIC:", PUBLIC_BLOCK_REASON)


## 10. Public inference (có chốt chặn)

In [ ]:
submission_path = None
if RUN_PUBLIC and PUBLIC_BLOCK_REASON is None:
    public_input = DATA_ROOT / "public_test/public_test.jsonl"
    PUBLIC_DIR.mkdir(parents=True, exist_ok=True)
    submission_path = PUBLIC_DIR / f"submission-{SELECTED}-{MODE}-{time.strftime('%Y%m%d-%H%M%S')}.jsonl"
    run([sys.executable, "scripts/run_inference.py",
         "--data", DATA_ROOT, "--split", "public_test", "--output", submission_path,
         "--adapter", SELECTED_ADAPTER, "--asr-cache", ASR_CACHE])

    with public_input.open(encoding="utf-8") as source:
        expected_ids = [json.loads(line)["id"] for line in source if line.strip()]
    with submission_path.open(encoding="utf-8") as source:
        rows = [json.loads(line) for line in source if line.strip()]
    predicted_ids = [r.get("id") for r in rows]
    if len(rows) != len(expected_ids) or len(set(predicted_ids)) != len(expected_ids):
        raise RuntimeError(f"Số dòng/ID không khớp: output={len(rows)}, input={len(expected_ids)}")
    if set(predicted_ids) != set(expected_ids):
        raise RuntimeError("ID trong submission không khớp public test.")
    if any(set(r) != {"id", "tool_calls", "state_hash"} for r in rows):
        raise RuntimeError("Submission sai schema; cần id, tool_calls, state_hash.")
    print(f"Submission hợp lệ: {submission_path} ({len(rows)} dòng; rỗng={sum(1 for r in rows if not r['tool_calls'])})")
elif PUBLIC_BLOCK_REASON is not None:
    print("Không tạo public submission:", PUBLIC_BLOCK_REASON)
else:
    print("RUN_PUBLIC=False: bỏ qua public inference.")


## 11. Đóng gói kết quả

In [ ]:
import tarfile

stamp = time.strftime("%Y%m%d-%H%M%S")
pack = WORKSPACE_DIR / f"lifemate-handoff-{MODE}-{stamp}.tar.gz"
include = [p for p in RUN_DIR.glob("dev_*") if p.is_file()]
if submission_path:
    include.append(submission_path)
include.append(RUN_DIR / "resume_preflight.json")
if CAND_DIR is not None:
    for name in ("final",):
        if (CAND_DIR / name).exists():
            include.append(CAND_DIR / name)
    newest = latest_checkpoint(CAND_DIR)
    if newest is not None:
        include.append(newest)   # có optimizer/scheduler để còn resume tiếp
    if (CAND_DIR / "preflight.json").is_file():
        include.append(CAND_DIR / "preflight.json")
with tarfile.open(pack, "w:gz") as archive:
    for path in include:
        archive.add(path, arcname=str(Path("lifemate") / path.relative_to(PROJECT_DIR)))
digest = hashlib.sha256(pack.read_bytes()).hexdigest()
(pack.parent / (pack.name + ".sha256")).write_text(f"{digest}  {pack.name}\n")
print("Gói kết quả:", pack, f"({pack.stat().st_size / 1024**2:.0f} MiB)")
print("SHA-256:", digest)
print("Tải file này về qua trình duyệt file của Jupyter trước khi huỷ instance.")


## Hoàn tất

- Adapter mới (nếu có): `lifemate_runs/vast-20261106-124816/warm_from_1524/final/` (hoặc `full/final/` ở chế độ `continue`).
- Báo cáo dev: `lifemate_runs/.../dev_*.json`; từng mẫu: `dev_*_samples.jsonl` (ASR vs gold, call đúng/sai, lý do dừng).
- Public submission: `lifemate_runs/.../public/submission-*.jsonl`.
- Checkpoint 1524 đã sao lưu ở `lifemate_runs/.../backup_checkpoint-1524/`.
- Gói tải về: `/workspace/lifemate-handoff-<mode>-<time>.tar.gz` (+ `.sha256`).

Sau khi nộp public và có điểm, ghi điểm cạnh tên file submission để lần sau đối chiếu đúng.
